# Landing — trust metadata

Source: `data/uk_investment_trusts.csv` (trust name, ticker, AIC sector, manager,
management group). Lands **exactly as it arrived** — no cleaning of any kind.

This file has two jobs. It supplies **manager and management group**, which is what
`dim_ticker` versions on in Gold, and its `ticker` column **defines the universe** the
Yahoo pull requests, so the list of trusts exists in one place only.

Expected: **120 rows**.

In [0]:
import os

import pandas as pd

CATALOG = "`index-vs-trust-pipeline`"
TABLE = f"{CATALOG}.landing.trusts_raw"

# Notebook runs from its own folder in a Git folder, so the repo root is two levels up.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CSV_PATH = os.path.join(REPO_ROOT, "data", "uk_investment_trusts.csv")

print(f"reading {CSV_PATH}")

In [0]:
# dtype=str keeps every column as text, which is what a CSV actually is.
# keep_default_na=False stops pandas turning blank cells into nulls -- that would be
# Landing quietly changing the data.
trusts = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

print(f"{len(trusts)} rows, {len(trusts.columns)} columns")
print(list(trusts.columns))
trusts.head(3)

In [0]:
sdf = spark.createDataFrame(trusts)
sdf.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print(f"wrote {TABLE}")

## Verification

In [0]:
%sql
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT ticker) AS distinct_tickers,
       SUM(CASE WHEN ticker = '' THEN 1 ELSE 0 END) AS blank_tickers
FROM `index-vs-trust-pipeline`.landing.trusts_raw;

Expect **120 rows, 119 distinct tickers, 2 blank** (Island Innovation, Witan). The 119
counts the empty string as one value: 118 real tickers plus the blank.
The blanks staying blank is the point: Landing did not drop them.